# Batch 4 — our own fine-tuning run (Part C)

**Base paper:** H. Rahman, *Fine-tuning Whisper for Pashto ASR: strategies and scale*, arXiv:2604.06507. Reproduction target: exp_009, **24.89 % WER / 8.075 % CER**.

Batch 3 asked *"is the paper's reported number verifiable?"* and answered it with the author's own weights. **This notebook asks the other question: does the training recipe reproduce at our compute budget?** Those are two different claims and the report keeps them apart.

### What to expect — read this before you start

You will **not** reach 24.89 %. The paper trained on 113.4 hours at a true batch of 64 in FP32; a free T4 gives you roughly a tenth of that data at an accumulated batch of 32 in fp16. Expect **35–45 % WER** at 12 hours of training audio.

That is a correct result, not a failed one, and the report is built to say so: `wer_vs_hours.png` turns the gap into a measured data-scaling trend rather than an apology. A fabricated match would fail the assignment; a well-explained 15-point gap does not.

### Design decisions, and why

| Decision | Why |
|---|---|
| Full fine-tuning, not LoRA | The paper measured LoRA r=64 at **+33.36 pp worse**. Faithful *and* correct. |
| **Dev split held out of training** | The paper merged dev into training and then ran its checkpoint-selection eval on the **test** split (recovered from its own log — see Batch 3 cell 3). We hold out a seeded dev subsample instead, so our selection never touches test. Costs us ~15 % of the rows the paper had, which works *against* us. |
| Checkpoints to Google Drive, resume supported | Colab disconnects. A lost 7-hour run costs a day. |
| Same filters, seed and decoding as Batch 3 | Otherwise Part B and Part C are not comparable, and cell 5 prints a fingerprint so you can prove the test split is identical. |
| A wall-clock guard that stops cleanly | Better to stop at 7 h with a saved best checkpoint than to be killed at 7.5 h with nothing. |

### Run order
Cells 1–6 are setup and data (~25 min). Cell 8 is the training run (6–9 h). Cells 9–12 are evaluation and figures (~1.5 h). **If you get disconnected, re-run cells 1–7 then cell 8 — it resumes from the last Drive checkpoint.**

## 1. Environment and GPU

Check which GPU you drew before committing to a long run. On a **T4** (Turing) use fp16. On a **P100** (Pascal) fp16 has no tensor cores and Whisper training can produce NaNs — prefer fp32 with a smaller batch, or restart the runtime until you get a T4. On an **A100** or **L4** (Ampere+) use bf16, which is both faster and numerically safer than fp16.

Cell 4 sets the precision automatically from what this cell detects.

In [ ]:
import os, sys, platform, subprocess, pathlib, json

ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
ON_KAGGLE = os.path.exists("/kaggle/input") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
PLATFORM  = "Google Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print("platform:", PLATFORM, "| python", sys.version.split()[0])

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi or "NO GPU -- Runtime > Change runtime type > T4 GPU, then re-run")

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
(RESULTS / "gpu.txt").write_text(smi or "no gpu", encoding="utf-8")

GPU_NAME = ""
for line in smi.split("\n"):
    for tag in ("T4", "P100", "V100", "A100", "L4", "A10"):
        if tag in line:
            GPU_NAME = tag; break
    if GPU_NAME: break
print("detected GPU family:", GPU_NAME or "unknown")
if GPU_NAME == "P100":
    print("\n*** P100: fp16 Whisper training is NaN-prone and has no tensor-core speedup.")
    print("*** Cell 4 will fall back to fp32 with a smaller batch. A T4 is a better draw.")

## 2. Pinned dependencies

Same pins as Batch 3, inside the floors the paper states. `datasets` must stay on 2.x.

Colab will ask you to restart the runtime. **Restart, then resume at cell 3** — do not re-run this cell.

In [ ]:
PINS = ["transformers==4.44.2", "datasets==2.21.0", "accelerate==0.33.0",
        "evaluate==0.4.3", "jiwer==3.0.4", "librosa==0.10.2", "soundfile==0.12.1"]

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
print(r.stdout[-2000:] or "(quiet)")
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-3000:])
print("\nRestart the runtime if asked, then resume at cell 3. Do NOT re-run this cell.")

In [ ]:
import sys, pathlib, subprocess
import torch, transformers, datasets, evaluate, jiwer
RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

props = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
env = "\n".join([
    f"platform              : {sys.platform}",
    f"python                : {sys.version.split()[0]}",
    f"torch                 : {torch.__version__}",
    f"cuda (torch)          : {torch.version.cuda}",
    f"gpu                   : {props.name if props else 'none'}",
    f"gpu_total_memory_gb   : {round(props.total_memory/1e9, 2) if props else 0}",
    f"gpu_capability        : {f'{props.major}.{props.minor}' if props else 'n/a'}",
    f"bf16_supported        : {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}",
    f"transformers          : {transformers.__version__}",
    f"datasets              : {datasets.__version__}",
    f"evaluate              : {evaluate.__version__}",
    f"jiwer                 : {jiwer.__version__}",
])
print(env)
(RESULTS / "environment.txt").write_text(env, encoding="utf-8")
with open(RESULTS / "pip_freeze.txt", "w", encoding="utf-8") as fh:
    fh.write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True).stdout)

## 3. Mount Google Drive — do not skip this

Checkpoints go to Drive, not to local disk. Colab wipes local disk on disconnect, and `whisper-small` checkpoints are about 1 GB each.

What goes where, and why:

| Location | Contents | Rationale |
|---|---|---|
| **Drive** | checkpoints, best model, `training_log.csv`, manifests | expensive to regenerate — must survive a disconnect |
| **local disk** | the decoded-audio cache (~1.5 GB) | cheap to regenerate in ~12 min, and writing it to Drive would be far slower |

On Kaggle there is no Drive; the cell falls back to `/kaggle/working`, which persists for the session. Commit the notebook to keep its output.

In [ ]:
import os, pathlib

RUN_TAG = "exp009_repro_tierB"      # change this if you run a second configuration

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = pathlib.Path("/content/drive/MyDrive/pashto_whisper")
elif ON_KAGGLE:
    DRIVE = pathlib.Path("/kaggle/working/pashto_whisper")
else:
    DRIVE = pathlib.Path("./pashto_whisper")

RUN_DIR   = DRIVE / RUN_TAG
AUDIO_CACHE = pathlib.Path("/content/audio_cache" if ON_COLAB else "./audio_cache")
for d in (DRIVE, RUN_DIR, AUDIO_CACHE):
    d.mkdir(parents=True, exist_ok=True)

print("persistent run dir :", RUN_DIR)
print("local audio cache  :", AUDIO_CACHE)

existing = sorted(RUN_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if existing:
    print(f"\nfound {len(existing)} checkpoint(s); latest = {existing[-1].name}")
    print("cell 8 will resume from it automatically")
else:
    print("\nno checkpoints yet -- this will be a fresh run")

## 4. Configuration

Every value carries the paper's setting beside it. Values marked **recovered** came from the author's own `experiment_log.txt`, which Batch 3 cell 3 downloaded — they are published fact, not assumption.

### The one judgement call: learning rate

The paper used **1e-4** at a true batch of 64 in FP32. We run an accumulated batch of 32 in fp16 on a tenth of the data, where 1e-4 tends to overfit or diverge. **1e-5** is the HuggingFace Whisper reference rate and the safer default.

This is a real deviation, so measure it rather than assume it: once the 1e-5 run finishes, if you have GPU time left, re-run with `LEARNING_RATE = 1e-4` and `RUN_TAG = "exp009_repro_lr1e4"`. That is ablation **E3a** in the report, and it converts "we changed the learning rate" into "we changed it and here is what it cost".

In [ ]:
import torch

# ===================== EDIT THIS CELL =====================
TRAIN_HOURS   = 12.0      # paper: 113.4 h (train+dev). Tier B. Try 25.0 for Tier C.
DEV_ROWS      = 1000      # held out of training. Paper: none held out (dev merged in).
LEARNING_RATE = 1e-5      # paper: 1e-4 (recovered). See the note above.
NUM_EPOCHS    = 10        # paper: 15 (recovered); early stopping is the real control
MAX_HOURS     = 7.0       # wall-clock guard: stop cleanly before Colab kills the session
SEED          = 42        # paper: 42
MODEL_NAME    = "openai/whisper-small"   # fallback: openai/whisper-base (paper exp_008, 27.13%)

# Batch: 8 x 4 = effective 32. Paper: true batch 64, accumulation 1 (recovered).
PER_DEVICE_BATCH = 8
GRAD_ACCUM       = 4
EVAL_BATCH       = 16

# ---- recovered from the author's experiment_log.txt (published fact) ----
WARMUP_STEPS       = 500     # recovered: warmup_steps 500, warmup_ratio 0.0
WEIGHT_DECAY       = 0.0     # recovered
EARLY_STOP_PATIENCE= 5       # recovered (the paper's text implies 3)
EVAL_STEPS         = 250     # paper: 500, but our run is far shorter
MIN_DURATION_S     = 0.5     # recovered (not the 1.0 used in our src/prepare_data.py)
MAX_DURATION_S     = 30.0    # recovered
OPTIM              = "adamw_torch"   # no optim is passed in the recovered config, so this default applied

# ---- decoding: MUST match Batch 3 or Part B and Part C are incomparable ----
NUM_BEAMS      = 1
MAX_NEW_TOKENS = 225
# ==========================================================

# precision chosen from the hardware, not guessed
BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
FP16 = bool(torch.cuda.is_available() and not BF16 and GPU_NAME != "P100")
if GPU_NAME == "P100":
    PER_DEVICE_BATCH, GRAD_ACCUM = 4, 8      # fp32 needs a smaller micro-batch
PRECISION = "bf16" if BF16 else ("fp16" if FP16 else "fp32")

PAPER = {"wer": 24.89, "wer_ortho": 27.73, "cer": 8.075, "zeroshot_wer": 120.4,
         "test_rows": 13791, "test_hours": 20.72, "train_hours": 113.4,
         "best_step": 8500, "best_epoch": 5.85, "lr": 1e-4, "effective_batch": 64,
         "base_wer": 27.13, "base_cer": 8.79}

print(f"model            {MODEL_NAME}")
print(f"train hours      {TRAIN_HOURS}      (paper {PAPER['train_hours']})")
print(f"learning rate    {LEARNING_RATE}    (paper {PAPER['lr']})")
print(f"effective batch  {PER_DEVICE_BATCH * GRAD_ACCUM}   "
      f"({PER_DEVICE_BATCH} x {GRAD_ACCUM})   (paper {PAPER['effective_batch']}, accumulation 1)")
print(f"precision        {PRECISION}        (paper fp32)")
print(f"epochs (max)     {NUM_EPOCHS}       (paper 15)")
print(f"wall-clock guard {MAX_HOURS} h")
print(f"decoding         greedy, max_new_tokens={MAX_NEW_TOKENS}  (same as Batch 3)")

cfg = {k: v for k, v in sorted(globals().items())
       if k.isupper() and isinstance(v, (int, float, str, bool))}
import json, pathlib
(pathlib.Path(RUN_DIR) / "effective_config.json").write_text(
    json.dumps(cfg, indent=2), encoding="utf-8")
print(f"\nwrote {RUN_DIR}/effective_config.json")

## 5. Data — identical filters, seed and splits to Batch 3

Set the same `DATA_ROUTE` you used in Batch 3. The filters and the seed are identical, so the test split this cell builds must be byte-identical to the one Batch 3 evaluated on.

The cell proves it rather than asserting it: it prints a **SHA-256 fingerprint of the ordered test file list**. Compare it against the fingerprint Batch 3 printed. If they differ, something changed and your Part B and Part C numbers are not comparable — fix that before training, not after.

Note the duration floor is now **0.5 s**, matching the author's recovered configuration rather than the 1.0 s in `src/prepare_data.py`. One fewer deviation to defend.

In [ ]:
# ===== set this to the SAME route you used in Batch 3 =====
DATA_ROUTE       = "B"
HF_DATASET_TOKEN = ""
KAGGLE_INPUT     = "/kaggle/input/common-voice-scripted-speech-24-0-pashto"
DRIVE_ARCHIVE    = "/content/drive/MyDrive/cv-corpus-24.0-ps.tar.gz"
# ==========================================================

import os, glob, csv as _csv, hashlib, json, pathlib, subprocess
import pandas as pd

CV_ROOT, hf_sets = None, {}

if DATA_ROUTE == "A":
    from datasets import load_dataset, Audio
    assert HF_DATASET_TOKEN, "paste a read token into HF_DATASET_TOKEN"
    for split in ("train", "dev", "test"):
        ds = load_dataset("ihanif/common-voice-pashto-24", split=split, token=HF_DATASET_TOKEN)
        hf_sets[split] = ds.cast_column("audio", Audio(sampling_rate=16000))
        print(f"{split:5s} {len(ds):>7} rows")

elif DATA_ROUTE == "B":
    hits = glob.glob(os.path.join(KAGGLE_INPUT, "**", "test.tsv"), recursive=True)
    assert hits, (f"no test.tsv under {KAGGLE_INPUT}\nAttach the Kaggle dataset "
                  "'ataullahaali/common-voice-scripted-speech-24-0-pashto' via Add Input.")
    CV_ROOT = pathlib.Path(hits[0]).parent

elif DATA_ROUTE == "C":
    if ON_COLAB:
        from google.colab import drive; drive.mount("/content/drive")
    assert os.path.exists(DRIVE_ARCHIVE), f"not found: {DRIVE_ARCHIVE}"
    os.makedirs("/content/cv", exist_ok=True)
    print("extracting the archive -- slow for a multi-gigabyte file")
    subprocess.run(["tar", "-xf", DRIVE_ARCHIVE, "-C", "/content/cv"], check=True)
    CV_ROOT = pathlib.Path(glob.glob("/content/cv/**/test.tsv", recursive=True)[0]).parent

if CV_ROOT:
    print("Common Voice root:", CV_ROOT)

In [ ]:
import numpy as np, soundfile as sf

def load_split_frame(split):
    """One code path for both data routes. Columns: audio, sentence, client_id, duration_s."""
    if DATA_ROUTE == "A":
        ds = hf_sets[split]
        df = pd.DataFrame({
            "sentence":  ds["sentence"],
            "client_id": ds["client_id"] if "client_id" in ds.column_names else ["unknown"] * len(ds),
            "audio":     list(range(len(ds))),
        })
        df["duration_s"] = np.nan          # HF route decodes lazily; no cheap duration source
        return df, ds

    df = pd.read_csv(CV_ROOT / f"{split}.tsv", sep="\t", quoting=_csv.QUOTE_NONE,
                     dtype=str, keep_default_na=False)
    df["audio"] = [str(CV_ROOT / "clips" / p) for p in df["path"]]

    dur_tsv = CV_ROOT / "clip_durations.tsv"
    if dur_tsv.exists():
        d = pd.read_csv(dur_tsv, sep="\t")
        durs = dict(zip(d[d.columns[0]], d[d.columns[1]] / 1000.0))
        df["duration_s"] = [durs.get(p, np.nan) for p in df["path"]]
    else:
        def probe(p):
            try:
                i = sf.info(p); return i.frames / i.samplerate
            except Exception:
                return np.nan
        df["duration_s"] = [probe(a) for a in df["audio"]]
    return df, None

def apply_filters(df, name):
    """Identical to Batch 3, with the recovered 0.5 s floor."""
    n0 = len(df)
    df = df[df["sentence"].str.strip() != ""]
    n1 = len(df)
    if df["duration_s"].notna().any():
        df = df[df["duration_s"].between(MIN_DURATION_S, MAX_DURATION_S, inclusive="both")]
    n2 = len(df)
    hrs = df["duration_s"].sum() / 3600.0 if df["duration_s"].notna().any() else float("nan")
    print(f"{name:6s} {n0:>7} -> {n1:>7} (empty text) -> {n2:>7} (duration)"
          f"   {hrs:>8.2f} h" if hrs == hrs else
          f"{name:6s} {n0:>7} -> {n1:>7} -> {n2:>7}")
    return df.reset_index(drop=True)

print("filters:")
train_df, train_hf = load_split_frame("train"); train_df = apply_filters(train_df, "train")
dev_df,   dev_hf   = load_split_frame("dev");   dev_df   = apply_filters(dev_df, "dev")
test_df,  test_hf  = load_split_frame("test");  test_df  = apply_filters(test_df, "test")

In [ ]:
# ---- seeded subsets, always WITHIN a split, never across -------------
def subset_hours(df, hours, seed):
    if df["duration_s"].notna().any():
        if df["duration_s"].sum() / 3600.0 <= hours:
            return df
        sh = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        return sh[sh["duration_s"].cumsum() / 3600.0 <= hours].reset_index(drop=True)
    approx = int(hours * 3600 / 4.4)          # CV24 Pashto averages ~4.4 s per clip
    return df.sample(n=min(approx, len(df)), random_state=seed).reset_index(drop=True)

train_full_rows, train_full_h = len(train_df), train_df["duration_s"].sum() / 3600.0
train_df = subset_hours(train_df, TRAIN_HOURS, SEED)
dev_df   = dev_df.sample(n=min(DEV_ROWS, len(dev_df)), random_state=SEED).reset_index(drop=True)

tr_h = train_df["duration_s"].sum() / 3600.0
print(f"train  {len(train_df):>6} rows  {tr_h:>7.2f} h   "
      f"(from {train_full_rows} / {train_full_h:.1f} h; paper used 92,880 / 113.4 h)")
print(f"dev    {len(dev_df):>6} rows   held OUT of training "
      f"(paper merged dev into training and evaluated on test)")
print(f"test   {len(test_df):>6} rows                      (paper 13,791)")
print(f"\nfraction of the paper's training audio: {100*tr_h/PAPER['train_hours']:.1f} % "
      f"-> about {PAPER['train_hours']/max(tr_h,1e-9):.1f}x less")

# ---- integrity checks -------------------------------------------------
if "client_id" in train_df:
    s = {k: set(v["client_id"]) for k, v in
         (("train", train_df), ("dev", dev_df), ("test", test_df))}
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        print(f"speaker-disjoint {a}-{b}: {not (s[a] & s[b])}")
print(f"dev/test identical-sentence overlap: "
      f"{len(set(dev_df['sentence']) & set(test_df['sentence']))}")

# ---- fingerprint: must equal the one Batch 3 printed -----------------
def fingerprint(df):
    key = "|".join(str(x) for x in df["sentence"])
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]

print(f"\nTEST SPLIT FINGERPRINT: {fingerprint(test_df)}")
print("Compare against Batch 3. If it differs, Part B and Part C are NOT comparable.")

for name, df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
    df.to_csv(RUN_DIR / f"manifest_{name}.csv", index=False, encoding="utf-8")
json.dump({"train_rows": len(train_df), "train_hours": round(float(tr_h), 3),
           "train_full_rows": train_full_rows, "dev_rows": len(dev_df),
           "test_rows": len(test_df), "seed": SEED,
           "test_fingerprint": fingerprint(test_df), "paper": PAPER},
          open(RUN_DIR / "dataset_stats.json", "w", encoding="utf-8"), indent=2)
print(f"manifests + dataset_stats.json written to {RUN_DIR}")

## 6. Decode the audio once, to a local cache

MP3 decoding is the bottleneck during training: Colab gives you about two CPU cores, and a T4 will sit idle waiting for them. So we decode every clip once to 16 kHz mono float16 `.npy` (≈140 KB per clip, ≈1.6 GB for 12 hours), then compute the log-Mel spectrogram on the fly in the collator, which is cheap.

Two reasons not to cache the log-Mel instead: a padded 80×3000 float32 spectrogram is 960 KB per clip — about 10 GB for this subset — and the mel transform is fast enough that caching it buys little.

**The cell is resumable.** It skips clips already decoded, so after a disconnect it finishes in seconds. Expect 10–15 minutes on a fresh run.

In [ ]:
import numpy as np, librosa, pathlib, hashlib, time
from concurrent.futures import ThreadPoolExecutor

AUDIO_CACHE = pathlib.Path(AUDIO_CACHE); AUDIO_CACHE.mkdir(parents=True, exist_ok=True)
SR = 16000

def cache_path(entry):
    return AUDIO_CACHE / (hashlib.md5(str(entry).encode()).hexdigest() + ".npy")

def decode_one(entry):
    out = cache_path(entry)
    if out.exists():
        return 0
    try:
        if DATA_ROUTE == "A":
            split, idx = entry.split(":")
            a = np.asarray(hf_sets[split][int(idx)]["audio"]["array"], dtype=np.float32)
        else:
            a, _ = librosa.load(str(entry), sr=SR, mono=True)   # 16 kHz mono
        np.save(out, a.astype(np.float16))
        return 1
    except Exception as e:
        print("FAILED", entry, e)
        return -1

# For route A the manifest holds integer indices, so tag them with their split
if DATA_ROUTE == "A":
    for split, df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
        df["audio"] = [f"{split}:{i}" for i in df["audio"]]

todo = list(train_df["audio"]) + list(dev_df["audio"]) + list(test_df["audio"])
print(f"{len(todo)} clips total; already cached: "
      f"{sum(cache_path(e).exists() for e in todo)}")

t0, done = time.time(), 0
with ThreadPoolExecutor(max_workers=8) as pool:
    for i, _ in enumerate(pool.map(decode_one, todo), 1):
        if i % 2000 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  {i}/{len(todo)}  {el:.0f}s elapsed  "
                  f"eta {el/i*(len(todo)-i):.0f}s", flush=True)

size_gb = sum(p.stat().st_size for p in AUDIO_CACHE.glob("*.npy")) / 1e9
print(f"\naudio cache: {len(list(AUDIO_CACHE.glob('*.npy')))} files, {size_gb:.2f} GB")

## 7. Dataset, collator, metrics and callbacks

- **Dataset** reads the cached waveform and computes the log-Mel spectrogram per item, so DataLoader workers parallelise it.
- **Collator** stacks the fixed 80×3000 features, pads labels and masks padding to `-100` so it contributes nothing to the cross-entropy. It also strips a leading BOS token that the model re-prepends, which would otherwise be counted twice.
- **Metrics** mirror Batch 3 exactly: normalised WER (the paper's headline metric and our checkpoint-selection criterion), orthographic WER, and CER.
- **`WallClockGuard`** stops training cleanly at `MAX_HOURS` so `load_best_model_at_end` still fires. Being killed by Colab at 7.5 hours with nothing saved is the failure mode this prevents.

Checkpoint selection uses **`metric_for_best_model="wer"`, not loss**. The author's own recovered curve shows validation loss bottoming at epoch 2.76 and then *rising* while WER keeps improving to its best at epoch 5.85 — selecting on loss would have picked a worse model.

In [ ]:
import numpy as np, torch, time, evaluate as hf_evaluate, jiwer
from torch.utils.data import Dataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          EarlyStoppingCallback, TrainerCallback)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="ps", task="transcribe")
_fe, _tok = processor.feature_extractor, processor.tokenizer
_norm = BasicTextNormalizer()
_wer, _cer = hf_evaluate.load("wer"), hf_evaluate.load("cer")

class PashtoASR(Dataset):
    def __init__(self, frame):
        self.entries = list(frame["audio"])
        self.sentences = list(frame["sentence"])
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        audio = np.load(cache_path(self.entries[i])).astype(np.float32)
        feats = _fe(audio, sampling_rate=SR).input_features[0]      # (80, 3000)
        return {"input_features": feats,
                "labels": _tok(self.sentences[i]).input_ids}

class Collator:
    def __call__(self, features):
        batch = {"input_features":
                 torch.tensor(np.stack([f["input_features"] for f in features]))}
        padded = _tok.pad([{"input_ids": f["labels"]} for f in features],
                          return_tensors="pt")
        labels = padded["input_ids"].masked_fill(padded.attention_mask.ne(1), -100)
        if (labels[:, 0] == _tok.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]            # the model re-prepends BOS
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    ids = pred.label_ids.copy()
    ids[ids == -100] = _tok.pad_token_id
    refs = _tok.batch_decode(ids, skip_special_tokens=True)
    hyps = _tok.batch_decode(pred.predictions, skip_special_tokens=True)
    rn, hn, ro, ho = [], [], [], []
    for r, h in zip(refs, hyps):
        a, b = _norm(r).strip(), _norm(h).strip()
        if not a:
            continue
        rn.append(a); hn.append(b or " "); ro.append(r); ho.append(h or " ")
    return {"wer":       100 * _wer.compute(references=rn, predictions=hn),
            "cer":       100 * _cer.compute(references=rn, predictions=hn),
            "wer_ortho": 100 * _wer.compute(references=ro, predictions=ho)}

class WallClockGuard(TrainerCallback):
    """Stop cleanly before Colab kills the session, so the best model is still saved."""
    def __init__(self, max_hours):
        self.limit = max_hours * 3600
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.limit:
            print(f"\n*** wall-clock guard: {self.limit/3600:.1f} h reached at step "
                  f"{state.global_step} -- stopping cleanly ***")
            control.should_training_stop = True
        return control

train_ds, dev_ds = PashtoASR(train_df), PashtoASR(dev_df)
print(f"train {len(train_ds)} | dev {len(dev_ds)}")
s = train_ds[0]
print(f"feature shape {np.shape(s['input_features'])} | label length {len(s['labels'])}")
print(f"decoded label: {_tok.decode(s['labels'], skip_special_tokens=True)[:90]}")

## 8. Train

6–9 hours at Tier B on a T4. **Leave the browser tab open** — Colab disconnects idle sessions.

If you are disconnected: re-run cells 1–7, then this cell. It finds the latest Drive checkpoint and resumes from it, including optimizer and scheduler state.

Watch the `eval_wer` column, not `eval_loss`. Loss can rise while WER keeps improving — that is exactly what happened in the author's own run.

**If you hit CUDA OOM:** set `PER_DEVICE_BATCH = 4` and `GRAD_ACCUM = 8` in cell 4 (same effective batch) and re-run from cell 4.

In [ ]:
import torch, time, json, pathlib
from transformers import set_seed
import numpy as np, random, os

# seeding -- paper used 42. Note this makes runs reproducible on the SAME hardware;
# cuDNN kernel selection and fp16 accumulation order still vary across GPU models.
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); set_seed(SEED)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "ps"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False               # required with gradient checkpointing

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_trainable/1e6:.2f}M  (paper reports 244M for whisper-small)")

args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    optim=OPTIM,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    fp16=FP16, bf16=BF16,
    gradient_checkpointing=True,
    group_by_length=False,       # length grouping makes batch composition data-order dependent
    dataloader_num_workers=2,
    predict_with_generate=True,
    generation_max_length=MAX_NEW_TOKENS,
    generation_num_beams=NUM_BEAMS,
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps",  save_steps=EVAL_STEPS,   # must equal eval_steps
    save_total_limit=2,                              # ~1 GB per checkpoint
    logging_steps=25,
    logging_dir=str(RUN_DIR / "logs"),
    load_best_model_at_end=True,
    metric_for_best_model="wer",                     # NOT loss -- see cell 7
    greater_is_better=False,
    report_to=["tensorboard"],
    seed=SEED, data_seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    args=args, model=model,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=Collator(), compute_metrics=compute_metrics,
    tokenizer=_fe,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
               WallClockGuard(MAX_HOURS)],
)

ckpts = sorted(RUN_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume = str(ckpts[-1]) if ckpts else None
print(f"\n{'resuming from ' + resume if resume else 'starting fresh'}")
print(f"precision={PRECISION}  effective batch={PER_DEVICE_BATCH*GRAD_ACCUM}  "
      f"lr={LEARNING_RATE}  eval every {EVAL_STEPS} steps\n")

t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
wall = time.time() - t0

BEST = RUN_DIR / "best"
trainer.save_model(str(BEST)); processor.save_pretrained(str(BEST))
print(f"\ntrained for {wall/3600:.2f} h -> {BEST}")

## 9. Export the training log and run metadata

`training_log.csv` is the source for Figures 1 and 2. `best_checkpoint.json` records the winning step and epoch, which is directly comparable to the paper's step 8500 / epoch 5.85.

In [ ]:
import pandas as pd, json, subprocess, torch, pathlib

hist, rows = trainer.state.log_history, {}
for e in hist:
    if e.get("step") is None:
        continue
    row = rows.setdefault(e["step"], {"step": e["step"]})
    for src, dst in (("loss", "train_loss"), ("eval_loss", "eval_loss"),
                     ("eval_wer", "eval_wer"), ("eval_cer", "eval_cer"),
                     ("eval_wer_ortho", "eval_wer_ortho"),
                     ("learning_rate", "learning_rate"),
                     ("grad_norm", "grad_norm"), ("epoch", "epoch")):
        if src in e:
            row[dst] = e[src]

log = pd.DataFrame(sorted(rows.values(), key=lambda r: r["step"]))
log.to_csv(RUN_DIR / "training_log.csv", index=False)

best = {"best_model_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric_wer": trainer.state.best_metric,
        "total_steps": trainer.state.global_step,
        "wall_clock_hours": round(wall / 3600, 3),
        "paper_best_step": PAPER["best_step"], "paper_best_epoch": PAPER["best_epoch"]}
if trainer.state.best_model_checkpoint:
    bs = int(pathlib.Path(trainer.state.best_model_checkpoint).name.split("-")[-1])
    best["best_step"] = bs
    m = log[log["step"] == bs]
    if not m.empty and "epoch" in m:
        best["best_epoch"] = round(float(m["epoch"].iloc[0]), 3)
json.dump(best, open(RUN_DIR / "best_checkpoint.json", "w", encoding="utf-8"), indent=2)

try:
    sha = subprocess.check_output(["git", "rev-parse", "HEAD"],
                                  stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    sha = "n/a (notebook run)"
json.dump({"run_tag": RUN_TAG, "git_commit": sha, "model": MODEL_NAME,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           "precision": PRECISION, "learning_rate": LEARNING_RATE,
           "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
           "train_hours": round(float(tr_h), 3), "seed": SEED,
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)
                           if torch.cuda.is_available() else None,
           "wall_clock_hours": round(wall / 3600, 3),
           "trainable_params_m": round(n_trainable / 1e6, 2)},
          open(RUN_DIR / "run_meta.json", "w", encoding="utf-8"), indent=2)

print(f"best WER on held-out dev: {best.get('best_metric_wer', float('nan')):.2f}% "
      f"at step {best.get('best_step')} (epoch {best.get('best_epoch')})")
print(f"paper's best: step {PAPER['best_step']}, epoch {PAPER['best_epoch']}")
print(f"total steps {best['total_steps']} | wall clock {best['wall_clock_hours']} h | "
      f"peak VRAM {round(torch.cuda.max_memory_allocated()/1e9,2)} GB")
log[log["eval_wer"].notna()][["step", "epoch", "eval_loss", "eval_wer", "eval_cer"]] \
    if "eval_wer" in log else log.tail()

## 10. Evaluate on the test split

Same decoding as Batch 3 — greedy, `max_new_tokens=225` — so the three systems (zero-shot, the author's checkpoint, ours) are directly comparable. The test split has not been touched until this cell.

~1.5 h for the full 13,791 utterances on a T4. Set `TEST_SUBSAMPLE` to e.g. 2000 for a faster first look, but the number that goes in the report should come from the full split, and §4.2 must say which.

In [ ]:
TEST_SUBSAMPLE = None     # None = full official test split

import torch, numpy as np, time, gc, pandas as pd, pathlib

eval_df = (test_df.sample(n=min(TEST_SUBSAMPLE, len(test_df)), random_state=SEED)
                  .reset_index(drop=True) if TEST_SUBSAMPLE else test_df)
print(f"evaluating on {len(eval_df)} utterances"
      f"{' (seeded subsample)' if TEST_SUBSAMPLE else ' (FULL official split)'}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
model_eval = WhisperForConditionalGeneration.from_pretrained(
    str(BEST), torch_dtype=DTYPE).to(DEVICE).eval()
model_eval.generation_config.forced_decoder_ids = None
model_eval.generation_config.language = "ps"
model_eval.generation_config.task = "transcribe"

hyps, t0 = [], time.time()
for start in range(0, len(eval_df), EVAL_BATCH):
    chunk = eval_df.iloc[start:start + EVAL_BATCH]
    audio = [np.load(cache_path(e)).astype(np.float32) for e in chunk["audio"]]
    feats = _fe(audio, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE, DTYPE)
    with torch.no_grad():
        ids = model_eval.generate(feats, language="ps", task="transcribe",
                                  num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS)
    hyps.extend(_tok.batch_decode(ids, skip_special_tokens=True))
    done = start + len(chunk)
    if start % (EVAL_BATCH * 25) == 0 or done == len(eval_df):
        el = time.time() - t0
        print(f"  {done:>6}/{len(eval_df)}  {el:>6.0f}s  "
              f"eta {el/max(done,1)*(len(eval_df)-done):>6.0f}s", flush=True)

hyps = [h.strip() for h in hyps]
refs = list(eval_df["sentence"])
del model_eval; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame({"reference": refs, "hypothesis": hyps}).to_csv(
    RUN_DIR / f"predictions_{RUN_TAG}_test.csv", index=False, encoding="utf-8")
print(f"\ndecoded in {(time.time()-t0)/60:.1f} min")

In [ ]:
import unicodedata, numpy as np, jiwer

PASHTO_DISTINCTIVE = set("ټډړڼږښځڅګۍ")
_CANON = {"ي": "ی", "ى": "ی", "ك": "ک",
          "ۀ": "ه", "ة": "ه"}
_STRIP = ("‌‍​ـًٌٍَُِّْ"
          "ٰ،؛؟۔")
_DIGITS = {**{chr(0x0660+i): str(i) for i in range(10)},
           **{chr(0x06F0+i): str(i) for i in range(10)}}

def pashto_canonical(t):
    for a, b in _CANON.items():
        t = t.replace(a, b)
    t = "".join(_DIGITS.get(c, c) for c in t if c not in _STRIP)
    return " ".join(unicodedata.normalize("NFC", t).split())

def score(refs, hyps, extra=None):
    rn, hn, ro, ho = [], [], [], []
    for r, h in zip(refs, hyps):
        if extra:
            r, h = extra(r), extra(h)
        a, b = _norm(r).strip(), _norm(h).strip()
        if not a:
            continue
        rn.append(a); hn.append(b or " "); ro.append(r); ho.append(h or " ")
    o = jiwer.process_words(rn, hn)
    err = o.substitutions + o.deletions + o.insertions
    return {"wer": 100*_wer.compute(references=rn, predictions=hn),
            "wer_ortho": 100*_wer.compute(references=ro, predictions=ho),
            "cer": 100*_cer.compute(references=rn, predictions=hn),
            "n": len(rn), "sub": o.substitutions, "del": o.deletions, "ins": o.insertions,
            "sub_pct": round(100*o.substitutions/err, 2) if err else 0,
            "del_pct": round(100*o.deletions/err, 2) if err else 0,
            "ins_pct": round(100*o.insertions/err, 2) if err else 0,
            "pashto_script_rate": 100*float(np.mean(
                [any(c in PASHTO_DISTINCTIVE for c in h) for h in hyps]))}

def bootstrap_ci(refs, hyps, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    c = []
    for r, h in zip(refs, hyps):
        a = _norm(r).strip()
        if not a:
            continue
        o = jiwer.process_words(a, _norm(h).strip() or " ")
        c.append((o.substitutions + o.deletions + o.insertions, len(a.split())))
    c = np.array(c, dtype=float)
    idx = rng.integers(0, len(c), size=(n_boot, len(c)))
    d = 100 * c[idx, 0].sum(axis=1) / c[idx, 1].sum(axis=1)
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

m_ours = score(refs, hyps)
ci_ours = bootstrap_ci(refs, hyps)
m_abl  = score(refs, hyps, extra=pashto_canonical)

print("=" * 76)
print(f"PART C -- our fine-tuned {MODEL_NAME}   N={m_ours['n']}")
print("=" * 76)
print(f"{'':<22}{'ours':>10}{'paper':>10}{'diff (pp)':>12}{'rel %':>10}")
for lab, k, ref in (("WER (normalised)", "wer", PAPER["wer"]),
                    ("WER (orthographic)", "wer_ortho", PAPER["wer_ortho"]),
                    ("CER", "cer", PAPER["cer"])):
    d = m_ours[k] - ref
    print(f"{lab:<22}{m_ours[k]:>10.2f}{ref:>10.3f}{d:>+12.2f}{100*d/ref:>+10.1f}")
print(f"\n95% bootstrap CI on WER: {ci_ours[0]:.2f} - {ci_ours[1]:.2f}")
print(f"errors: S {m_ours['sub']} / D {m_ours['del']} / I {m_ours['ins']}"
      f"   substitutions {m_ours['sub_pct']}% of errors (paper reports ~70-78%)")
print(f"Pashto-script output rate: {m_ours['pashto_script_rate']:.2f}%")
print(f"WER/CER ratio: {m_ours['wer']/m_ours['cer']:.2f}  "
      f"(paper {PAPER['wer']/PAPER['cer']:.2f})")
print(f"\nnormalisation ablation (E3b): {m_ours['wer']:.2f}% -> {m_abl['wer']:.2f}%  "
      f"({m_ours['wer']-m_abl['wer']:.2f} pp is Pashto spelling variation)")
print(f"\nvs the zero-shot baseline you measured in Batch 3: fill in the delta and the")
print(f"paired bootstrap p-value using results/predictions_zeroshot_small.csv")

## 11. Write metrics.csv and the §7.1 arithmetic

Appends our row to the same `metrics.csv` schema Batch 3 used, with `source="ours"`. Paste the printed block into **§7.1** of the report.

In [ ]:
import pandas as pd, torch, json, pathlib

row = {"tag": RUN_TAG, "source": "ours", "experiment": "fine-tuned (ours)",
       "model": MODEL_NAME, "split": f"cv24-test (N={m_ours['n']})",
       "n_utterances": m_ours["n"],
       "wer": round(m_ours["wer"], 2), "wer_ortho": round(m_ours["wer_ortho"], 2),
       "cer": round(m_ours["cer"], 3),
       "wer_ci_lo": round(ci_ours[0], 2), "wer_ci_hi": round(ci_ours[1], 2),
       "paper_wer": PAPER["wer"], "paper_cer": PAPER["cer"],
       "difference_pp": round(m_ours["wer"] - PAPER["wer"], 2),
       "difference_rel_pct": round(100*(m_ours["wer"]-PAPER["wer"])/PAPER["wer"], 1),
       "substitutions": m_ours["sub"], "deletions": m_ours["del"],
       "insertions": m_ours["ins"], "sub_pct_of_errors": m_ours["sub_pct"],
       "del_pct_of_errors": m_ours["del_pct"], "ins_pct_of_errors": m_ours["ins_pct"],
       "pashto_script_rate": round(m_ours["pashto_script_rate"], 2),
       "wer_pashto_canon": round(m_abl["wer"], 2),
       "num_beams": NUM_BEAMS, "precision": PRECISION,
       "learning_rate": LEARNING_RATE,
       "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
       "train_hours": round(float(tr_h), 3),
       "total_steps": best["total_steps"], "best_step": best.get("best_step"),
       "best_epoch": best.get("best_epoch"),
       "wall_clock_h": best["wall_clock_hours"],
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "eval_subsample": TEST_SUBSAMPLE or "full split", "seed": SEED}

path = RUN_DIR / "metrics.csv"
frame = pd.DataFrame([row])
frame.to_csv(path, mode="a", header=not path.exists(), index=False, encoding="utf-8")
frame.to_csv(RESULTS / f"metrics_{RUN_TAG}.csv", index=False, encoding="utf-8")

print("=" * 76)
print("SECTION 7.1 -- PART C")
print("=" * 76)
print("Sign convention: difference = ours - paper (positive means ours is worse).\n")
print(f"  WER_paper           = {PAPER['wer']:.2f} %")
print(f"  WER_ours            = {m_ours['wer']:.2f} %   95% CI "
      f"[{ci_ours[0]:.2f}, {ci_ours[1]:.2f}]")
print(f"  Absolute difference = {m_ours['wer']:.2f} - {PAPER['wer']:.2f} = "
      f"{m_ours['wer']-PAPER['wer']:+.2f} pp")
print(f"  Relative difference = {100*(m_ours['wer']-PAPER['wer'])/PAPER['wer']:+.1f} %")
print(f"  CER_paper = {PAPER['cer']:.3f} %   CER_ours = {m_ours['cer']:.2f} %   "
      f"difference = {m_ours['cer']-PAPER['cer']:+.2f} pp")
print(f"\nTraining budget")
print(f"  our training audio  = {tr_h:.2f} h   paper = {PAPER['train_hours']} h   "
      f"-> {PAPER['train_hours']/max(tr_h,1e-9):.1f}x less")
print(f"  our optimizer steps = {best['total_steps']}   paper best step = {PAPER['best_step']}")
print(f"  our effective batch = {PER_DEVICE_BATCH*GRAD_ACCUM}   paper = "
      f"{PAPER['effective_batch']} (accumulation 1)")
print(f"  our precision       = {PRECISION}   paper = fp32")
print(f"  our learning rate   = {LEARNING_RATE}   paper = {PAPER['lr']}")
print("=" * 76)
frame.T

## 12. Figures 1, 2 and 7

- **Figure 1** — training loss, with the selected checkpoint marked.
- **Figure 2** — validation loss and validation WER on twin axes. Check whether the two decouple the way the author's own curve does; if they do, say so in §9, because it reproduces a structural property of the paper rather than just a number.
- **Figure 7** — WER against training hours, with the paper's 113.4 h / 24.89 % point marked. **This is the figure that explains your gap.** It needs at least two of our own runs at different `TRAIN_HOURS`; with one run it plots a single point plus the paper's, which is still usable but much weaker.

Figures 3–6 come from the Batch 3 notebook.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pandas as pd, pathlib, glob

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)
log = pd.read_csv(RUN_DIR / "training_log.csv")
best_step = best.get("best_step")

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS / f"{name}.{ext}", dpi=300, bbox_inches="tight")
        fig.savefig(RUN_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig); print("  ", name)

print("figures:")

# --- Figure 1 --------------------------------------------------------
tr = log.dropna(subset=["train_loss"]) if "train_loss" in log else pd.DataFrame()
if not tr.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tr["step"], tr["train_loss"], color="#2b6cb0", lw=1.3)
    if best_step:
        ax.axvline(best_step, ls="--", color="#c53030", lw=1,
                   label=f"selected checkpoint (step {best_step})")
        ax.legend(frameon=False)
    ax.set_xlabel("Training step"); ax.set_ylabel("Training loss (cross-entropy)")
    ax.set_title(f"Training loss -- {RUN_TAG}\n"
                 f"{tr_h:.1f} h of audio, lr {LEARNING_RATE}, effective batch "
                 f"{PER_DEVICE_BATCH*GRAD_ACCUM}, {PRECISION}")
    ax.grid(alpha=0.3)
    save(fig, "training_loss")

# --- Figure 2 --------------------------------------------------------
ev = log.dropna(subset=["eval_loss"]) if "eval_loss" in log else pd.DataFrame()
if not ev.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(ev["step"], ev["eval_loss"], color="#2b6cb0", marker="o", ms=4,
            lw=1.3, label="Validation loss")
    ax.set_xlabel("Training step"); ax.set_ylabel("Validation loss", color="#2b6cb0")
    ax.tick_params(axis="y", colors="#2b6cb0")
    if "eval_wer" in ev:
        tw = ax.twinx()
        tw.plot(ev["step"], ev["eval_wer"], color="#c53030", marker="s", ms=4,
                lw=1.3, label="Validation WER")
        tw.axhline(PAPER["wer"], ls=":", color="#4a5568", lw=1.1,
                   label=f"paper's reported WER ({PAPER['wer']}%)")
        tw.set_ylabel("Validation WER (%)", color="#c53030")
        tw.tick_params(axis="y", colors="#c53030")
        lines = ax.get_lines() + tw.get_lines()
        ax.legend(lines, [l.get_label() for l in lines], frameon=False, fontsize=8)
        i_loss, i_wer = ev["eval_loss"].idxmin(), ev["eval_wer"].idxmin()
        print(f"     lowest validation loss at step {ev.loc[i_loss,'step']} "
              f"(WER {ev.loc[i_loss,'eval_wer']:.2f})")
        print(f"     best validation WER  at step {ev.loc[i_wer,'step']} "
              f"(loss {ev.loc[i_wer,'eval_loss']:.4f})")
        if ev.loc[i_loss, "step"] != ev.loc[i_wer, "step"]:
            print("     -> loss and WER DECOUPLE, as in the author's own run. Discuss in section 9.")
    if best_step:
        ax.axvline(best_step, ls="--", color="#718096", lw=1)
    ax.set_title("Validation loss and WER (checkpoint selected on WER, not loss)")
    ax.grid(alpha=0.3)
    save(fig, "validation_loss")

In [ ]:
# --- Figure 7: WER vs training hours, across every run you have -------
import glob, pandas as pd, pathlib

rows = []
for p in sorted(glob.glob(str(DRIVE / "*" / "metrics.csv"))):
    try:
        rows.append(pd.read_csv(p))
    except Exception:
        pass
allruns = (pd.concat(rows, ignore_index=True)
             .drop_duplicates(subset=["tag"], keep="last")
             .sort_values("train_hours")) if rows else pd.DataFrame()

if len(allruns):
    print(allruns[["tag", "train_hours", "wer", "cer", "learning_rate",
                   "effective_batch", "precision"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4.8))
if len(allruns) >= 2:
    ax.plot(allruns["train_hours"], allruns["wer"], marker="o", ms=7, lw=1.6,
            color="#2b6cb0", label="This reproduction")
elif len(allruns) == 1:
    ax.scatter(allruns["train_hours"], allruns["wer"], marker="o", s=90,
               color="#2b6cb0", label="This reproduction (single run)")

for _, r in allruns.iterrows():
    # below-right, so the topmost point's label cannot collide with the title
    ax.annotate(f"{r['wer']:.1f}%\n{r['train_hours']:.0f} h",
                (r["train_hours"], r["wer"]), textcoords="offset points",
                xytext=(10, -4), va="top", fontsize=8, color="#1a365d")

ax.scatter([PAPER["train_hours"]], [PAPER["wer"]], marker="*", s=320,
           color="#c53030", zorder=5,
           label=f"Rahman (2026), reported ({PAPER['train_hours']} h, {PAPER['wer']}%)")
ax.annotate(f"{PAPER['wer']}%\n{PAPER['train_hours']} h",
            (PAPER["train_hours"], PAPER["wer"]), textcoords="offset points",
            xytext=(-14, 12), ha="right", fontsize=8, color="#742a2a")

# a log-linear fit through our runs, extrapolated to the paper's data budget.
# It shows whether our gap is simply where the data-scaling trend predicts.
if len(allruns) >= 2:
    import numpy as np
    coef = np.polyfit(np.log10(allruns["train_hours"]), allruns["wer"], 1)
    xs = np.logspace(np.log10(allruns["train_hours"].min()),
                     np.log10(PAPER["train_hours"]), 50)
    ax.plot(xs, np.polyval(coef, np.log10(xs)), ls="--", lw=1.1,
            color="#718096", label="log-linear trend of our runs, extrapolated")
    pred = np.polyval(coef, np.log10(PAPER["train_hours"]))
    print(f"\n     trend extrapolated to {PAPER['train_hours']} h predicts "
          f"{pred:.1f}% WER; the paper reports {PAPER['wer']}%")
    print("     Report this in section 9.3, and read it honestly:")
    if abs(pred - PAPER["wer"]) < 4:
        print("       the extrapolation lands close to the paper, so our gap is")
        print("       what the data-scaling trend predicts -- the method reproduces,")
        print("       and only the data budget differs")
    elif pred < PAPER["wer"]:
        print("       the extrapolation undershoots the paper's WER, so a naive")
        print("       log-linear fit over two or three small runs is optimistic:")
        print("       scaling curves flatten. Say that rather than claiming we would")
        print("       have beaten the paper")
    else:
        print("       the extrapolation overshoots the paper's WER, so data volume")
        print("       alone does not account for the whole gap -- the learning rate,")
        print("       batch size and precision deviations take the remainder")

ax.set_xscale("log")
lo = min(allruns["train_hours"].min() if len(allruns) else PAPER["train_hours"],
         PAPER["train_hours"])
ax.set_xlim(lo * 0.55, PAPER["train_hours"] * 2.2)   # pad, or the star is clipped
ax.set_xlabel("Training audio (hours, log scale)"); ax.set_ylabel("WER (%)")
ax.set_title("WER against training-data volume\n"
             "quantifies the compute-induced gap rather than asserting it")
ax.legend(frameon=False, fontsize=8.5); ax.grid(alpha=0.3, which="both")
save(fig, "wer_vs_hours")

if len(allruns) < 2:
    print("\nOnly one run found. For a real trend line, re-run this notebook with")
    print("  TRAIN_HOURS = 3.0   and   RUN_TAG = 'exp009_repro_tierA'")
    print("which takes about 1.5 h and gives Figure 7 a second point. If you keep")
    print("only one run, delete Figure 7 from section 8 and say so in section 9.3.")

## 13. Sample predictions, and optionally publish the checkpoint

Our weights are about 1 GB, too large for Google Classroom. Pushing them to the Hugging Face Hub is the cleanest way to make the reproduction complete — §9.6 of the report has a placeholder for the URL. It is optional; "available on request" is an acceptable alternative.

In [ ]:
import jiwer, pandas as pd, pathlib

per = []
for r, h in zip(refs, hyps):
    a = _norm(r).strip()
    if not a:
        continue
    o = jiwer.process_words(a, _norm(h).strip() or " ")
    per.append({"reference": r, "hypothesis": h, "wer_utt": round(100*o.wer, 1),
                "n_sub": o.substitutions, "n_del": o.deletions, "n_ins": o.insertions})
per = pd.DataFrame(per).sort_values("wer_utt").reset_index(drop=True)

k, mid = 7, len(per) // 2
sample = pd.concat([per.head(k).assign(bucket="best"),
                    per.iloc[mid-3:mid+4].assign(bucket="median"),
                    per.tail(k).assign(bucket="worst")])
sample.insert(0, "tag", RUN_TAG)
sample.to_csv(RESULTS / f"sample_predictions_{RUN_TAG}.csv", index=False, encoding="utf-8")
per.to_csv(RUN_DIR / f"per_utterance_{RUN_TAG}.csv", index=False, encoding="utf-8")

print(f"per-utterance WER: min {per.wer_utt.min():.1f}  median {per.wer_utt.median():.1f}  "
      f"mean {per.wer_utt.mean():.1f}  max {per.wer_utt.max():.1f}")
print(f"transcribed perfectly: {(per.wer_utt==0).sum()} of {len(per)} "
      f"({100*(per.wer_utt==0).mean():.1f}%)")
print("\nSet the transcript columns to Noto Naskh Arabic and right-to-left in Word,")
print("or the Pashto prints as empty boxes.")
sample[["bucket", "wer_utt", "reference", "hypothesis"]].head(21)

In [ ]:
# ---- optional: publish the fine-tuned weights to the Hugging Face Hub ----
PUSH_TO_HUB = False
HF_WRITE_TOKEN = ""                       # a WRITE token from huggingface.co/settings/tokens
HUB_REPO = "your-username/whisper-small-pashto-cs444-repro"

if PUSH_TO_HUB:
    assert HF_WRITE_TOKEN, "paste a write token into HF_WRITE_TOKEN"
    from huggingface_hub import HfApi
    api = HfApi(token=HF_WRITE_TOKEN)
    api.create_repo(HUB_REPO, exist_ok=True)

    card = f"""---
language: ps
license: apache-2.0
base_model: {MODEL_NAME}
---

# whisper-small Pashto -- CS-444 reproduction of Rahman (2026)

Reproduction of exp_009 from H. Rahman, "Fine-tuning Whisper for Pashto ASR:
strategies and scale", arXiv:2604.06507.

| | ours | paper (exp_009) |
|---|---|---|
| WER (normalised) | {m_ours['wer']:.2f} % | {PAPER['wer']} % |
| CER | {m_ours['cer']:.2f} % | {PAPER['cer']} % |
| training audio | {tr_h:.1f} h | {PAPER['train_hours']} h |
| learning rate | {LEARNING_RATE} | {PAPER['lr']} |
| effective batch | {PER_DEVICE_BATCH*GRAD_ACCUM} | {PAPER['effective_batch']} |
| precision | {PRECISION} | fp32 |

Evaluated on the Common Voice v24 Pashto test split (N={m_ours['n']}), greedy
decoding, max_new_tokens={MAX_NEW_TOKENS}. The gap is dominated by training-data
volume: we used about {100*tr_h/PAPER['train_hours']:.0f} % of the paper's audio.

Trained for a university assignment. Dataset: Mozilla Common Voice 24.0 Pashto, CC0-1.0.
"""
    (pathlib.Path(BEST) / "README.md").write_text(card, encoding="utf-8")
    api.upload_folder(folder_path=str(BEST), repo_id=HUB_REPO)
    print(f"pushed -> https://huggingface.co/{HUB_REPO}")
else:
    print("PUSH_TO_HUB is False. Set it to True with a write token to publish the weights,")
    print("or write 'available on request' in section 9.6 of the report.")

## 14. Zip and download

Everything needed for the report. The weights stay on Drive; this archive holds the logs, metrics, predictions and figures.

In [ ]:
import shutil, os, sys, pathlib, glob

STAGE = pathlib.Path("batch4_bundle"); shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir()
for pattern in ("training_log.csv", "best_checkpoint.json", "run_meta.json",
                "effective_config.json", "dataset_stats.json", "metrics.csv",
                "manifest_*.csv", "predictions_*.csv", "per_utterance_*.csv",
                "*.png", "*.pdf"):
    for p in glob.glob(str(RUN_DIR / pattern)):
        shutil.copy(p, STAGE)
for p in glob.glob("results/*"):
    if os.path.isfile(p):
        shutil.copy(p, STAGE)

print("bundle contents:")
for p in sorted(STAGE.iterdir()):
    print(f"  {p.stat().st_size:>12,}  {p.name}")

shutil.make_archive("batch4_results", "zip", STAGE)
print(f"\nbatch4_results.zip  {os.path.getsize('batch4_results.zip'):,} bytes")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("batch4_results.zip")
else:
    print("Kaggle: find it in /kaggle/working via the Output tab.")

## 15. What to fill in the report

| Report location | From |
|---|---|
| §6.3 Table 2 | `training_log.csv` — one row per evaluation checkpoint |
| §6.3 summary table | cell 11 — training hours, data ratio, steps, wall clock, peak VRAM |
| §4.3 training rows/hours/speakers, best step and epoch | cells 5 and 9 |
| §5.1 GPU, VRAM, versions | `results/environment.txt`, `results/gpu.txt` |
| §5.3 the placeholders for what you actually ran | `effective_config.json` — learning rate, precision, batch composition, epochs, warmup |
| §6.4 Table 4 (normalisation ablation, our row) | cell 10 |
| §7 Table 5 Part C column, §7.1, Abstract, §10 | cell 11 |
| §8 Figures 1, 2, 7 | `results/*.png` |
| §8.1 sample predictions | `sample_predictions_<tag>.csv` |
| §9.3 gap decomposition | cell 11's training-budget block |
| §9.6 checkpoint availability | cell 13 — the Hub URL, or "available on request" |
| Appendix A.1 log excerpts | printed output of cells 5, 8, 9, 10, 11 |
| Screenshots 5 and 6 | cells 8 and 10 |

### Write §9.3 from measurements, not adjectives

Cell 11 prints every quantity the gap decomposition needs. Rank them: training-data volume first (with Figure 7 as evidence), then learning rate, then optimizer-step count, then effective batch and precision, then the held-out dev split, then the re-implemented trainer. Every one of those is now a number you measured rather than a factor you asserted.

And state the positive result plainly: WER fell from the zero-shot baseline you measured in Batch 3 to this, which is a large, statistically significant improvement, and it confirms the paper's central claim that Pashto needs weight updates rather than better prompting — even though our absolute number does not match.

### Two optional runs, in order of value

1. **`TRAIN_HOURS = 3.0`, `RUN_TAG = "exp009_repro_tierA"`** — about 1.5 h, and it gives Figure 7 a real trend line instead of a single point. Best value per GPU-hour of anything left.
2. **`LEARNING_RATE = 1e-4`, `RUN_TAG = "exp009_repro_lr1e4"`** — ablation E3a. Converts "we changed the learning rate" into a measured cost. Expect it to be worse at this data scale; that is the finding.

### If training failed or ran out of time

Submit with §6.3 marked "not completed" and explain it in §9.5 Limitations. Batch 3 already gives you a verified reproduction of the paper's headline number, a measured zero-shot baseline, both original analyses and five figures. That is a complete report. This notebook upgrades it from "we verified the paper" to "we also retrained it" — valuable, but not what the grading hinges on.